## Curso: Análise e Desenvolvimento de Sistemas

## Disciplina: DISRUPTIVE ARCHITECTURES: IOT, IOB & GENERATIVE AI

## Turma: 2TDSPS

## Professor: André Tritiack

## Projeto: FIAP Connect — IA de Compatibilidade de Grupos

### Integrantes:
| Nome | RM |
|------|----|
| Alexis Rondo | 560384 |
| Vinicius Rodrigues de Oliveira | 559611 |

---

## Definição do Problema

O **FIAP Connect** é uma plataforma para formação inteligente de grupos para o Challenge Sprint da FIAP.

Atualmente, o sistema usa regras fixas no banco Oracle para calcular compatibilidade entre alunos e grupos, baseado nas 7 disciplinas do Challenge:
- MOBILE, JAVA, DEVOPS, BD, DOTNET, IOT, SQA

O problema é que essa lógica é **determinística** — ela apenas verifica quantas disciplinas o aluno cobre, sem considerar outros fatores que influenciam o sucesso da formação do grupo.

### Proposta de IA

Treinar um **modelo de classificação** (Machine Learning supervisionado) que analise múltiplos atributos de um aluno e de um grupo para prever o **nível de compatibilidade**: `ALTA`, `MEDIA` ou `BAIXA`.

### Justificativa da escolha do modelo

Utilizamos **Random Forest** e **KNN** por serem algoritmos de classificação supervisionada adequados para:
- Datasets tabulares com atributos numéricos
- Problemas de classificação multiclasse
- Fácil interpretação dos resultados

Esses modelos foram estudados em aula e são eficientes para o volume de dados do projeto.

### Estratégia de integração com Oracle APEX (Sprint 4)

Na Sprint 4, o modelo treinado será salvo em `.pkl` e disponibilizado via **API Flask**. O Oracle APEX chamará essa API via REST, enviando os dados do aluno e do grupo, e receberá a predição de compatibilidade para exibir no aplicativo mobile.

## 1. Importação de bibliotecas

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

## 2. Criação do dataset

Os dados simulam combinações de alunos e grupos do FIAP Connect, baseados na mesma estrutura do banco Oracle APEX.

**Atributos (features):**
- `num_habilidades_aluno` — quantas disciplinas o aluno domina (1 a 7)
- `num_habilidades_faltantes_grupo` — quantas disciplinas ainda faltam no grupo (0 a 7)
- `num_cobertas` — quantas disciplinas do aluno coincidem com as que o grupo precisa
- `percentual_cobertura` — num_cobertas / num_habilidades_faltantes_grupo * 100
- `mesmo_periodo` — 1 se aluno e grupo são do mesmo período (MANHA/NOITE), 0 caso contrário
- `mesma_unidade` — 1 se aluno e grupo são da mesma unidade (PAULISTA/ACLIMACAO), 0 caso contrário
- `vagas_disponiveis` — vagas restantes no grupo (1 ou 2)

**Variável alvo:**
- `compatibilidade` — ALTA, MEDIA ou BAIXA

In [ ]:
# Gerando dados sintéticos baseados nas regras reais do FIAP Connect
np.random.seed(42)

registros = []

for _ in range(300):
    num_hab_aluno = np.random.randint(1, 8)        # 1 a 7 disciplinas
    num_faltantes = np.random.randint(1, 8)         # 1 a 7 faltando no grupo
    num_cobertas = np.random.randint(0, min(num_hab_aluno, num_faltantes) + 1)
    percentual = round((num_cobertas / num_faltantes) * 100, 1) if num_faltantes > 0 else 0
    mesmo_periodo = np.random.choice([0, 1], p=[0.3, 0.7])
    mesma_unidade = np.random.choice([0, 1], p=[0.4, 0.6])
    vagas = np.random.choice([1, 2])

    # Regras de classificação baseadas na lógica real do FIAP Connect
    if num_cobertas == num_faltantes and mesmo_periodo == 1 and mesma_unidade == 1:
        compat = 'ALTA'
    elif num_cobertas == num_faltantes and mesmo_periodo == 1:
        compat = 'ALTA'
    elif percentual >= 50 and mesmo_periodo == 1:
        compat = 'MEDIA'
    elif percentual >= 30 and mesma_unidade == 1:
        compat = 'MEDIA'
    else:
        compat = 'BAIXA'

    registros.append([
        num_hab_aluno, num_faltantes, num_cobertas,
        percentual, mesmo_periodo, mesma_unidade, vagas, compat
    ])

colunas = [
    'num_habilidades_aluno', 'num_habilidades_faltantes_grupo',
    'num_cobertas', 'percentual_cobertura',
    'mesmo_periodo', 'mesma_unidade', 'vagas_disponiveis',
    'compatibilidade'
]

df = pd.DataFrame(registros, columns=colunas)
print(f'Dataset criado com {df.shape[0]} registros e {df.shape[1]} atributos.')

## 3. Análise Exploratória dos Dados

In [ ]:
# Visualizar os primeiros registros
df.head(10)

In [ ]:
# Dimensões do dataframe
df.shape

In [ ]:
# Informações dos atributos
df.info()

In [ ]:
# Estatísticas descritivas
df.describe()

In [ ]:
# Distribuição da variável alvo
df['compatibilidade'].value_counts()

## 4. Preparação dos dados

In [ ]:
# Conversão da variável categórica para numérica
le = LabelEncoder()
df['compatibilidade_cod'] = le.fit_transform(df['compatibilidade'])

# Verificando o mapeamento
print('Mapeamento das classes:')
for i, classe in enumerate(le.classes_):
    print(f'  {classe} -> {i}')

In [ ]:
# Visualização após transformação
df[['compatibilidade', 'compatibilidade_cod']].head(10)

### Criação das variáveis X e y

- **X** — dataframe com os 7 atributos (features)
- **y** — array com a variável alvo (compatibilidade codificada)

In [ ]:
# Features (X)
X = df[['num_habilidades_aluno', 'num_habilidades_faltantes_grupo',
        'num_cobertas', 'percentual_cobertura',
        'mesmo_periodo', 'mesma_unidade', 'vagas_disponiveis']]

X.head()

In [ ]:
# Variável alvo (y)
y = df['compatibilidade_cod'].values
y

## 5. Separação dos dados (70% treino / 30% teste)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f'Dados de treino: {X_train.shape[0]} registros')
print(f'Dados de teste:  {X_test.shape[0]} registros')

## 6. Treinamento do Modelo 1 — Random Forest

In [ ]:
# Instância do modelo
modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)

In [ ]:
# Treinamento
modelo_rf.fit(X_train, y_train)

In [ ]:
# Predições
y_predict_rf = modelo_rf.predict(X_test)
y_predict_rf

In [ ]:
# Acurácia do Random Forest
acc_rf = accuracy_score(y_test, y_predict_rf)
print(f'Acurácia Random Forest: {acc_rf:.4f} ({acc_rf*100:.2f}%)')

In [ ]:
# Relatório de classificação
print('Relatório de Classificação - Random Forest:\n')
print(classification_report(y_test, y_predict_rf, target_names=le.classes_))

## 7. Treinamento do Modelo 2 — KNN

In [ ]:
# Instância do modelo
modelo_knn = KNeighborsClassifier(n_neighbors=5)

In [ ]:
# Treinamento
modelo_knn.fit(X_train, y_train)

In [ ]:
# Predições
y_predict_knn = modelo_knn.predict(X_test)
y_predict_knn

In [ ]:
# Acurácia do KNN
acc_knn = accuracy_score(y_test, y_predict_knn)
print(f'Acurácia KNN: {acc_knn:.4f} ({acc_knn*100:.2f}%)')

In [ ]:
# Relatório de classificação
print('Relatório de Classificação - KNN:\n')
print(classification_report(y_test, y_predict_knn, target_names=le.classes_))

## 8. Comparação dos modelos

In [ ]:
print('=' * 40)
print('COMPARAÇÃO DE ACURÁCIA')
print('=' * 40)
print(f'Random Forest: {acc_rf*100:.2f}%')
print(f'KNN:           {acc_knn*100:.2f}%')
print('=' * 40)

melhor = 'Random Forest' if acc_rf >= acc_knn else 'KNN'
print(f'\nMelhor modelo: {melhor}')
print(f'\nEsse modelo será salvo em .pkl e usado na API Flask na Sprint 4.')

## 9. Simulação com dados reais do Oracle APEX

Vamos testar o modelo com dados que espelham os usuários reais cadastrados no banco Oracle APEX do FIAP Connect.

**Cenário:** A aluna Beatriz (RM333333) busca um grupo. Ela domina IOT e SQA.
- **Grupo 1 (FIAP Connect Team):** faltam IOT e SQA → ela cobre tudo
- **Grupo 2 (Oracle Innovators):** faltam MOBILE, BD, DOTNET, IOT, SQA → ela cobre 2 de 5

In [ ]:
# Dados reais do APEX - Beatriz buscando grupo
dados_apex = pd.DataFrame([
    # Beatriz vs Grupo 1 (FIAP Connect Team) - deve ser ALTA
    {
        'num_habilidades_aluno': 2,          # IOT, SQA
        'num_habilidades_faltantes_grupo': 2, # faltam IOT e SQA
        'num_cobertas': 2,                    # cobre as 2
        'percentual_cobertura': 100.0,
        'mesmo_periodo': 1,                   # NOITE = NOITE
        'mesma_unidade': 1,                   # PAULISTA = PAULISTA
        'vagas_disponiveis': 1
    },
    # Beatriz vs Grupo 2 (Oracle Innovators) - deve ser MEDIA
    {
        'num_habilidades_aluno': 2,
        'num_habilidades_faltantes_grupo': 5,
        'num_cobertas': 2,
        'percentual_cobertura': 40.0,
        'mesmo_periodo': 1,
        'mesma_unidade': 1,
        'vagas_disponiveis': 2
    }
])

print('Dados de entrada (simulando requisição do APEX):')
dados_apex

In [ ]:
# Predição usando o melhor modelo
melhor_modelo = modelo_rf if acc_rf >= acc_knn else modelo_knn

predicoes = melhor_modelo.predict(dados_apex)
labels = le.inverse_transform(predicoes)

print('Resultados da predição de compatibilidade:')
print(f'  Beatriz → Grupo 1 (FIAP Connect Team):  {labels[0]}')
print(f'  Beatriz → Grupo 2 (Oracle Innovators):   {labels[1]}')
print()
print('Esses resultados são consistentes com a lógica de negócio:')
print('- Grupo 1: Beatriz cobre 100% das disciplinas faltantes → ALTA')
print('- Grupo 2: Beatriz cobre 40% das faltantes → MEDIA')

## 10. Salvamento do modelo (preparação para Sprint 4)

Na Sprint 4, o modelo será exportado em `.pkl` e carregado pela API Flask para servir predições via HTTP.

In [ ]:
import pickle

# Salvando o melhor modelo
with open('modelo_compatibilidade.pkl', 'wb') as f:
    pickle.dump(melhor_modelo, f)

# Salvando o LabelEncoder para decodificar as predições
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print('Modelo salvo em: modelo_compatibilidade.pkl')
print('Encoder salvo em: label_encoder.pkl')
print()
print('Na Sprint 4, a API Flask carregará esses arquivos para servir predições.')

## 11. Diagrama de Integração (Sprint 4)

```
┌─────────────────┐    ┌──────────────────┐    ┌────────────────────┐
│   App Mobile    │    │   Oracle APEX    │    │   Oracle Database  │
│  (React Native) │───▶│   (REST API)     │───▶│   (Tabelas:        │
│                 │    │                  │    │    USUARIO,         │
│  Tela de busca  │    │  GET /grupos-    │    │    GRUPO,           │
│  de grupos      │    │  compativeis/:rm │    │    HABILIDADE...)   │
└─────────────────┘    └────────┬─────────┘    └────────────────────┘
                                │
                                │ POST /predict
                                ▼
                       ┌──────────────────┐
                       │   API Flask      │
                       │                  │
                       │  modelo.pkl      │
                       │  (sklearn)       │
                       │                  │
                       │  Retorna JSON:   │
                       │  {"predicao":    │
                       │   "ALTA"}        │
                       └──────────────────┘
```

### Fluxo de funcionamento:

1. O aluno abre o app mobile e acessa a tela de busca de grupos
2. O app chama a API REST do Oracle APEX
3. O APEX consulta o banco Oracle para obter os dados do aluno e dos grupos disponíveis
4. O APEX monta os atributos (features) e envia via POST para a API Flask
5. A API Flask carrega o modelo `.pkl` e executa a predição
6. O Flask retorna o nível de compatibilidade (ALTA, MEDIA, BAIXA) em JSON
7. O APEX recebe a resposta e retorna os grupos ordenados por compatibilidade
8. O app mobile exibe os resultados para o aluno

## 12. Conclusão — Resultados Parciais (Sprint 3)

Nesta Sprint 3, concluímos a etapa de **planejamento e validação** do componente de IA:

- **Problema definido:** predição de compatibilidade aluno-grupo no FIAP Connect
- **Modelo escolhido:** Random Forest (classificação supervisionada com sklearn)
- **Dados identificados:** atributos derivados das tabelas USUARIO, GRUPO, HABILIDADE, GRUPO_USUARIO_HABILIDADE do Oracle APEX
- **Diagrama de integração:** IA ↔ Flask ↔ APEX ↔ Oracle Database
- **Demonstração funcional simulada:** modelo treinado com dados sintéticos, testado com cenários reais do APEX

### Próximos passos (Sprint 4):
1. Salvar modelo em `.pkl`
2. Criar API Flask com endpoint `/predict`
3. Empacotar em Docker
4. Integrar com Oracle APEX via REST
5. Testar no app mobile